# HRNet-W32 joint fine-tuning

Fine-tune the ImageNet-1k HRNet-W32 body and the selected local linear head together on stored fold 0. Keep the 384-pixel letterbox and fixed 0.25 threshold. This notebook calls the experiment CLI so the same recovery and export code works outside Jupyter. The selected-fold score is a screening result, not an independent ensemble gain.


In [ ]:
from pathlib import Path
import json
import math
import time
import os
import random
import shutil
import subprocess
import sys

import numpy as np

SEED = 42
RUN_ID = "hrnet_w32_finetune_fold0_seed42"
MODEL_NAME = "hrnet_w32.ms_in1k"
FOLD = 0
IMAGE_SIZE = 384
THRESHOLD = 0.25
BATCH_SIZE = 2
EFFECTIVE_BATCH_SIZE = 16
MAX_EPOCHS = 20
PATIENCE = 4
PRECISION = "16-mixed"  # Use "bf16-mixed" if the Colab GPU supports it.

DRIVE_ROOT = Path("/content/drive/MyDrive/AI301-stage2")
INPUT_ROOT = DRIVE_ROOT / "hrnet_w32_input"
DRIVE_OUTPUT = DRIVE_ROOT / "hrnet_w32_runs"
WORK_ROOT = Path("/content/hrnet_w32_work")
HEAD_DIR = WORK_ROOT / "artifacts/hrnet_head/hrnet_w32_head_fold0_seed42"
RESUME_CHECKPOINT = None  # Set to a restored resume_epoch_XX.ckpt when recovering.
EXPORT_ONLY = False  # Set True with RESUME_CHECKPOINT after training ended but export failed.

session_started = time.monotonic()
random.seed(SEED)
np.random.seed(SEED)
if EFFECTIVE_BATCH_SIZE % BATCH_SIZE:
    raise ValueError("Effective batch size must be divisible by physical batch size")
print(f"{MODEL_NAME}, fold {FOLD}, 384 px, batch {BATCH_SIZE} × {EFFECTIVE_BATCH_SIZE // BATCH_SIZE} accumulation")


## Mount Drive and stage inputs

Run `uv run --frozen python scripts/stage_hrnet_colab_inputs.py` locally and upload its `artifacts/hrnet_w32_input/` folder to `MyDrive/AI301-stage2/hrnet_w32_input/`. It contains the branch code, locked environment, dataset, selected local head, and source manifest. Frozen feature shards stay local. This is the only interactive authentication step. The run writes epoch backups and final artifacts under `hrnet_w32_runs/`.


In [ ]:
from google.colab import drive, userdata

drive.mount("/content/drive")
for required in (INPUT_ROOT / "repo/stage2/hrnet_finetune.py", INPUT_ROOT / "repo/uv.lock",
                 INPUT_ROOT / "repo/folds.csv", INPUT_ROOT / "data/train.csv",
                 INPUT_ROOT / "data/test.csv", INPUT_ROOT / "data/sample_submission.csv",
                 INPUT_ROOT / "head/head_best.pt", INPUT_ROOT / "head/head_best.json",
                 INPUT_ROOT / "head/backbone.json", INPUT_ROOT / "manifest.json"):
    if not required.is_file():
        raise FileNotFoundError(f"Missing Colab input: {required}")
if not WORK_ROOT.exists():
    shutil.copytree(INPUT_ROOT / "repo", WORK_ROOT,
                    ignore=shutil.ignore_patterns(".git", ".venv", "artifacts", "__pycache__", ".env"))
    shutil.copytree(INPUT_ROOT / "data", WORK_ROOT / "data")
    shutil.copy2(INPUT_ROOT / "manifest.json", WORK_ROOT / "source_manifest.json")
    HEAD_DIR.mkdir(parents=True)
    for name in ("head_best.pt", "head_best.json", "backbone.json"):
        shutil.copy2(INPUT_ROOT / "head" / name, HEAD_DIR / name)
else:
    print("Using existing runtime workspace:", WORK_ROOT)
manifest = json.loads((WORK_ROOT / "source_manifest.json").read_text())
import hashlib
for relative, expected_hash in manifest["files"].items():
    group, _, remainder = relative.partition("/")
    staged = (WORK_ROOT / remainder if group == "repo" else
              HEAD_DIR / remainder if group == "head" else WORK_ROOT / "data" / remainder)
    digest = hashlib.sha256(staged.read_bytes()).hexdigest()
    if digest != expected_hash:
        raise ValueError(f"Staged input failed checksum: {staged}")
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
try:
    wandb_key = userdata.get("WANDB_API_KEY")
except Exception:
    wandb_key = None
if wandb_key:
    os.environ["WANDB_API_KEY"] = wandb_key
print("Inputs staged; W&B online logging:", bool(wandb_key))


## Environment and preflight

Install the locked environment, verify the fixed split and every image, then check one CUDA optimizer update and measure one complete epoch before fixing the full run schedule. A failed smoke test leaves no training run directory.


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
subprocess.run(["uv", "sync", "--frozen", "--python", "3.12"], cwd=WORK_ROOT, check=True)
base_command = ["uv", "run", "--frozen", "python", "-m", "stage2.hrnet_finetune",
                "--root", str(WORK_ROOT), "--head-dir", str(HEAD_DIR),
                "--run-id", RUN_ID, "--batch-size", str(BATCH_SIZE),
                "--max-epochs", str(MAX_EPOCHS), "--precision", PRECISION]
subprocess.run([*base_command, "--preflight-only"], cwd=WORK_ROOT, check=True)


In [ ]:
subprocess.run([*base_command, "--smoke-only"], cwd=WORK_ROOT, check=True)


## Choose a run length from measured throughput

A complete train and validation epoch measures this GPU and data path. The profile is archived to Drive. The notebook reserves 45 minutes for prediction export and allows 30% margin on each training epoch, then sets the cosine schedule horizon before the actual run.


In [ ]:
profile_path = WORK_ROOT / "artifacts/hrnet_profile" / f"{RUN_ID}.json"
if RESUME_CHECKPOINT is None:
    subprocess.run([*base_command, "--profile-epoch"], cwd=WORK_ROOT, check=True)
    profile = json.loads(profile_path.read_text())
    drive_profile = DRIVE_OUTPUT / RUN_ID / "profile.json"
    drive_profile.parent.mkdir(parents=True, exist_ok=True)
    if not drive_profile.exists():
        shutil.copy2(profile_path, drive_profile)
    if hashlib.sha256(drive_profile.read_bytes()).hexdigest() != hashlib.sha256(profile_path.read_bytes()).hexdigest():
        raise IOError("Drive profile copy failed checksum verification")
    remaining_seconds = 5 * 3600 - (time.monotonic() - session_started)
    available_training_seconds = remaining_seconds - 45 * 60
    planned_epochs = min(MAX_EPOCHS, math.floor(available_training_seconds / (1.30 * profile["full_epoch_seconds"])))
    if planned_epochs < 1:
        raise RuntimeError("This GPU cannot fit one joint-training epoch plus export in the remaining five-hour budget")
    run_budget_hours = remaining_seconds / 3600
    print(f"Measured {profile['full_epoch_seconds'] / 60:.1f} min/epoch, "
          f"peak {profile['peak_allocated_gib']:.2f} GiB; plan {planned_epochs} epochs")
else:
    previous = json.loads((DRIVE_OUTPUT / RUN_ID / "results/run.json").read_text())
    planned_epochs = previous["config"]["max_epochs"]
    run_budget_hours = previous["config"]["budget_hours"]
    print(f"Resuming existing {planned_epochs}-epoch schedule")
base_command[base_command.index("--max-epochs") + 1] = str(planned_epochs)
base_command.extend(["--budget-hours", str(run_budget_hours)])


## Joint fine-tuning and export

The CLI starts from the local head, trains body and head with warm-up and cosine decay, keeps BatchNorm running statistics fixed, and selects the best epoch by fold-0 macro F1. Every completed epoch gets a full resume checkpoint and checksum-verified Drive copy. To resume after a VM reset, set `RESUME_CHECKPOINT` to a Drive `resume_epoch_XX.ckpt`; this cell restores the exact best checkpoint recorded in that resume state. Set `EXPORT_ONLY=True` if training has ended and only prediction export needs recovery. The planned horizon comes from the measured full epoch above. Validation and test CSVs contain independent sigmoid probabilities for all five labels; matching logits are saved separately. The source notebook is archived with results, while Colab saves the executed notebook through its normal Drive save. Nothing is submitted to Kaggle.


In [ ]:
command = [*base_command, "--drive-dir", str(DRIVE_OUTPUT)]
if RESUME_CHECKPOINT is not None:
    drive_run = DRIVE_OUTPUT / RUN_ID
    local_run = WORK_ROOT / "artifacts/hrnet_finetune" / RUN_ID
    checkpoint_dir = local_run / "checkpoints"
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    resume_source = Path(RESUME_CHECKPOINT)
    if not resume_source.is_file():
        raise FileNotFoundError(resume_source)
    import torch
    resume_state = torch.load(resume_source, map_location="cpu", weights_only=False)
    selection = resume_state["callbacks"]["hrnet_selected_checkpoint"]
    best_source = drive_run / "checkpoints" / f"best_epoch_{selection['best_epoch']:02d}.ckpt"
    if not best_source.is_file():
        raise FileNotFoundError(f"Missing selected checkpoint for resume: {best_source}")
    def digest(path):
        with path.open("rb") as stream:
            return hashlib.file_digest(stream, "sha256").digest()
    def restore(source, destination):
        if source.resolve() == destination.resolve():
            return
        if destination.exists():
            if digest(source) != digest(destination):
                raise FileExistsError(f"Existing restored file differs: {destination}")
            return
        shutil.copy2(source, destination)
        if digest(source) != digest(destination):
            raise IOError(f"Restored file failed checksum: {destination}")
    restore(drive_run / "results/run.json", local_run / "run.json")
    restore(best_source, checkpoint_dir / best_source.name)
    resume_local = checkpoint_dir / resume_source.name
    restore(resume_source, resume_local)
    command.extend(["--resume", str(resume_local)])
if EXPORT_ONLY:
    if RESUME_CHECKPOINT is None:
        raise ValueError("EXPORT_ONLY needs RESUME_CHECKPOINT")
    command.append("--export-only")
subprocess.run(command, cwd=WORK_ROOT, check=True)


## Results

Read the selected checkpoint record and compact per-label metrics. The CSVs in Drive are the inputs for local threshold and ensemble analysis.


In [ ]:
run_dir = WORK_ROOT / "artifacts/hrnet_finetune" / RUN_ID
results = json.loads((run_dir / "results.json").read_text())
print("Selected epoch:", results["selected_epoch"])
print("Checkpoint SHA-256:", results["checkpoint_sha256"])
print(f"Macro F1: {results['metrics']['macro_f1']:.5f}; macro AP: {results['metrics']['macro_ap']:.5f}")
for label in ("Cat", "Dog", "Moris", "Motya", "Biatrix"):
    print(f"{label:8s} F1 {results['metrics']['per_label_f1'][label]:.5f}  AP {results['metrics']['per_label_ap'][label]:.5f}")
import csv
with (run_dir / "history.csv").open(newline="") as stream:
    selected_history = next(row for row in csv.DictReader(stream) if int(row["epoch"]) == results["selected_epoch"])
print("Selected-epoch validation BCE by label:",
      {label: round(float(selected_history[f"bce_{label}"]), 5) for label in ("Cat", "Dog", "Moris", "Motya", "Biatrix")})
print("Drive:", DRIVE_OUTPUT / RUN_ID)
print("Probability files:", results["files"]["validation"]["probabilities"]["path"],
      results["files"]["test"]["probabilities"]["path"])


## Next step

Download the completed checkpoint and prediction CSVs. Compare HRNet with existing members on the same validation IDs, then tune thresholds locally without retraining. The selected fold-0 score alone cannot establish an improvement over the current public score.
